# QSVM Water Potability — Quantinuum Nexus / H2 Emulator
### Challenge 2 · Quantathon CR 2026 · Feature map personalizado (domain-informed)

Este cuaderno corre **de arriba a abajo en Quantinuum Nexus** y produce solo los
entregables **analíticos del modelo** (no EDA del dataset).

Pipeline:

1. Carga del CSV local (80 filas) y split estratificado 80/20 congelado.
2. Imputación por **mediana por clase** ajustada **solo en train** (sin fuga).
3. Estandarización + codificación angular `theta = (pi/2)*tanh(z)` **por fold**.
4. Baseline clásico SVM-RBF con **GridSearchCV 5-fold** (C, gamma de la rúbrica).
5. Kernel cuántico de fidelidad con el **feature map personalizado** (RY-RZ + RZZ disperso 0-1, 1-3, 2-3).
6. Validación del circuito de overlap contra el statevector exacto.
7. Kernel exacto (statevector) → **CV QSVM** → métricas pareadas contra RBF.
8. **Kernel con shots en el emulador H2 de Nexus** (batch de circuitos, checkpoints).
9. Test reservado evaluado **una sola vez** al final, con las 5 métricas + matriz de confusión.
10. Auditoría de recursos compilados para H2 (2Q gates, profundidad, equivalencia numérica).

Todos los bloques largos imprimen progreso incremental con tiempos y ETA.

## Bloque 0 — Entorno y verificación de dependencias

In [1]:
# ============================================================
# BLOQUE 0 — VERIFICACIÓN DEL ENTORNO (Nexus web)
# ============================================================
# En Nexus el entorno viene gestionado. NO se instala nada:
# un `pip install -U` puede romper las versiones ya provistas.

import importlib, platform, sys

print("Python:", platform.python_version())
print("Ejecutable:", sys.executable)
print("-" * 58)

faltan = []
for module_name in ["pytket", "qnexus", "numpy", "pandas",
                    "sklearn", "matplotlib", "scipy"]:
    try:
        m = importlib.import_module(module_name)
        v = getattr(m, "__version__", "(sin __version__)")
        print(f"[OK]     {module_name:12s} {v}")
    except ImportError:
        print(f"[FALTA]  {module_name}")
        faltan.append(module_name)

if faltan:
    print("\nFaltan paquetes:", faltan)
    print("En Nexus web NO use pip desde el notebook. Use una terminal del Lab:")
    print("   pip install --user " + " ".join(faltan))
else:
    print("\nEntorno completo. No hace falta instalar nada.")

Python: 3.11.10
Ejecutable: /opt/conda/bin/python
----------------------------------------------------------
[OK]     pytket       2.16.0
[OK]     qnexus       (sin __version__)
[OK]     numpy        2.2.6
[OK]     pandas       2.2.3
[OK]     sklearn      1.5.2
[OK]     matplotlib   3.9.2
[OK]     scipy        1.16.3

Entorno completo. No hace falta instalar nada.


In [2]:
# ============================================================
# BLOQUE 0B — IMPORTS Y SEMILLAS
# ============================================================

import os, json, time, random, hashlib, platform, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # Nexus no siempre tiene backend interactivo
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    GridSearchCV,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    make_scorer,
)

from scipy.stats import wilcoxon

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------- Semillas congeladas ----------------
SEED = 4524
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

print("SEED global      :", SEED)
print("Python           :", platform.python_version())
print("NumPy            :", np.__version__)
print("Pandas           :", pd.__version__)
print("scikit-learn     :", sklearn.__version__)
print("Timestamp        :", datetime.now().isoformat(timespec="seconds"))

SEED global      : 4524
Python           : 3.11.10
NumPy            : 2.2.6
Pandas           : 2.2.3
scikit-learn     : 1.5.2
Timestamp        : 2026-07-24T12:07:46


## Bloque 1 — Configuración congelada

Todo parámetro del experimento vive acá. **Nada se re-tunea más adelante**:
así se evita el error común de *comparación injusta* y de *selección con el test*.

In [3]:
# ============================================================
# BLOQUE 1 — CONFIGURACIÓN CONGELADA
# ============================================================

# ---- Rutas -------------------------------------------------
# En Nexus web el CSV subido queda en el directorio de trabajo del notebook.
# Si el nombre difiere, ajústelo acá. El Bloque 2 lista los CSV visibles.
DATA_PATH  = Path("water_potability_80.csv")
OUTPUT_DIR = Path("outputs_qsvm_h2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Variables ---------------------------------------------
FEATURES = ["ph", "Hardness", "Sulfate", "Conductivity"]
TARGET   = "Potability"

QUBIT_MAP = {0: "ph", 1: "Hardness", 2: "Sulfate", 3: "Conductivity"}
N_QUBITS  = 4

# ---- Split -------------------------------------------------
TEST_SIZE = 0.20

# ---- Feature map personalizado -----------------------------
#   (0,1) pH x Dureza      (1,3) Dureza x Conductividad
#   (2,3) Sulfato x Conductividad
CUSTOM_EDGES = [(0, 1), (1, 3), (2, 3)]

# ---- Codificación angular ----------------------------------
ANGLE_SCALE = np.pi / 2.0          # theta = ANGLE_SCALE * tanh(z)

# ---- SVM ---------------------------------------------------
QSVM_C        = 1.0                # fijo para el kernel cuántico
QSVM_CLASS_WT = None
RBF_PARAM_GRID = {                 # grilla exigida por la rúbrica
    "svc__C":     [0.1, 1, 10],
    "svc__gamma": ["scale", "auto", 0.01],
}

# ---- Validación cruzada ------------------------------------
CV_SPLITS  = 5
CV_REPEATS = 4                     # 20 particiones

# ---- Dispositivo Nexus -------------------------------------
# NOMBRES VÁLIDOS verificados con qnx.devices.get_all():
#   "H2-1LE"       emulador local, SIN ruido, no consume cuota -> para depurar
#   "H2-Emulator"  emulador CON noise_specs reales             -> corrida final
#   "H1-1LE" / "H1-Emulator"  equivalentes de H1
H2_DEVICE   = "H2-Emulator"
SHOT_LEVELS = [256, 1024]
OPT_LEVEL   = 2

# ---- Tamaño del experimento cuántico -----------------------
# 16 -> 136 circuitos ; 12 -> 78 ; 8 -> 36
AUDIT_N = 12

# ---- Flags derivados de la corrida -------------------------
IS_NOISY_RUN      = not H2_DEVICE.endswith("LE")
FRESH_CHECKPOINTS = True

# ---- PSD ---------------------------------------------------
PSD_EPSILON = 1e-8

print("CONFIGURACIÓN CONGELADA")
print("-" * 62)
print("CSV                :", DATA_PATH)
print("Salidas            :", OUTPUT_DIR.resolve())
print("Features           :", FEATURES)
print("Qubits             :", N_QUBITS, "->", QUBIT_MAP)
print("Aristas custom     :", CUSTOM_EDGES)
print("Escala angular     : (pi/2)*tanh(z)")
print("CV                 :", f"{CV_SPLITS}-fold x {CV_REPEATS} repeticiones",
      f"= {CV_SPLITS*CV_REPEATS} particiones")
print("Grilla RBF         :", RBF_PARAM_GRID)
print("C del QSVM         :", QSVM_C, "(sin tuning)")
print("Dispositivo Nexus  :", H2_DEVICE)
print("Shots              :", SHOT_LEVELS)
print("AUDIT_N            :", AUDIT_N,
      f"-> {AUDIT_N*(AUDIT_N+1)//2} circuitos por nivel de shots")

if H2_DEVICE == "H2-1LE":
    print("\nMODO DEPURACIÓN: H2-1LE es el emulador sin ruido y no consume")
    print("cuota. Cambie a 'H2-Emulator' solo para la corrida final.")
else:
    print(f"\nATENCIÓN: {H2_DEVICE} consume cuota real de la cuenta.")

# ---- Limpieza de checkpoints de corridas anteriores --------
if FRESH_CHECKPOINTS:
    import shutil
    shutil.rmtree(CKPT_DIR, ignore_errors=True)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    print("\nCheckpoints borrados (los de H2-1LE / AUDIT_N=16 ya no aplican).")
else:
    _viejos = sorted(CKPT_DIR.glob("jobref_*.json"))
    if _viejos:
        print(f"\nAVISO: {len(_viejos)} checkpoints presentes; el Bloque 12")
        print("intentará reengancharse a esos jobs en vez de enviar nuevos.")
        for p in _viejos:
            print("   ", p.name)

CONFIGURACIÓN CONGELADA
--------------------------------------------------------------
CSV                : water_potability_80.csv
Salidas            : /home/jovyan/outputs_qsvm_h2
Features           : ['ph', 'Hardness', 'Sulfate', 'Conductivity']
Qubits             : 4 -> {0: 'ph', 1: 'Hardness', 2: 'Sulfate', 3: 'Conductivity'}
Aristas custom     : [(0, 1), (1, 3), (2, 3)]
Escala angular     : (pi/2)*tanh(z)
CV                 : 5-fold x 4 repeticiones = 20 particiones
Grilla RBF         : {'svc__C': [0.1, 1, 10], 'svc__gamma': ['scale', 'auto', 0.01]}
C del QSVM         : 1.0 (sin tuning)
Dispositivo Nexus  : H2-1LE
Shots              : [256, 1024]
AUDIT_N            : 16 -> 136 circuitos por nivel de shots

MODO DEPURACIÓN: H2-1LE es el emulador sin ruido y no consume
cuota. Cambie a 'H2-Emulator' solo para la corrida final.


## Bloque 2 — Carga del CSV y split estratificado 80/20

El split se hace **antes** de cualquier imputación o escalado. El conjunto de test
queda apartado y no se toca hasta el Bloque 12.

In [4]:
# ============================================================
# BLOQUE 2 — CARGA Y SPLIT 80/20
# ============================================================

# ---- Localizar el CSV --------------------------------------
if not DATA_PATH.exists():
    print("No se encontró:", DATA_PATH)
    print("Directorio actual:", Path.cwd())
    print("\nCSV visibles desde acá:")
    encontrados = sorted(Path.cwd().rglob("*.csv"))
    for p in encontrados:
        print("   ", p.relative_to(Path.cwd()))
    if not encontrados:
        print("    (ninguno)")
    raise FileNotFoundError(
        f"Ajuste DATA_PATH en el Bloque 1 con uno de los nombres de arriba."
    )

df_raw = pd.read_csv(DATA_PATH)
print("CSV cargado:", DATA_PATH.name, "->", df_raw.shape)
print("Columnas   :", df_raw.columns.tolist())

# ---- Verificación de columnas ------------------------------
faltantes = [c for c in FEATURES + [TARGET] if c not in df_raw.columns]
assert not faltantes, f"Faltan columnas en el CSV: {faltantes}"

df = df_raw[FEATURES + [TARGET]].copy()
df[TARGET] = df[TARGET].astype(int)

# Filas sin etiqueta no sirven para nada supervisado
n_antes = len(df)
df = df[df[TARGET].isin([0, 1])].reset_index(drop=True)
if len(df) != n_antes:
    print(f"AVISO: se descartaron {n_antes - len(df)} filas sin etiqueta válida.")

# ID trazable para reproducibilidad
df.insert(0, "Sample_ID", [f"W_{i:03d}" for i in range(1, len(df) + 1)])

print("\nFilas usables      :", len(df))
print("Distribución clases:")
print(df[TARGET].value_counts().sort_index().to_string())

print("\nNaN por variable (antes de imputar):")
print(df[FEATURES].isna().sum().to_string())

# ---- Split estratificado 80/20 -----------------------------
idx_all = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx_all,
    test_size=TEST_SIZE,
    stratify=df[TARGET],
    random_state=SEED,
    shuffle=True,
)

df_train = df.iloc[idx_train].reset_index(drop=True)
df_test  = df.iloc[idx_test].reset_index(drop=True)

X_train_raw = df_train[FEATURES].copy()
y_train     = df_train[TARGET].to_numpy(dtype=int)
X_test_raw  = df_test[FEATURES].copy()
y_test      = df_test[TARGET].to_numpy(dtype=int)

# ---- Controles anti-fuga -----------------------------------
assert set(df_train["Sample_ID"]).isdisjoint(set(df_test["Sample_ID"])), \
    "FUGA: hay Sample_ID compartidos entre train y test."
assert len(df_train) + len(df_test) == len(df)

print("\nSPLIT 80/20 (estratificado, seed fija)")
print("-" * 60)
print("Train:", X_train_raw.shape, "| clases:",
      np.bincount(y_train, minlength=2).tolist())
print("Test :", X_test_raw.shape,  "| clases:",
      np.bincount(y_test,  minlength=2).tolist())
print("Intersección de IDs: 0  (verificado)")

df_train.to_csv(OUTPUT_DIR / "split_train.csv", index=False)
df_test.to_csv(OUTPUT_DIR / "split_test.csv",   index=False)
print("\nSplit guardado en", OUTPUT_DIR)

CSV cargado: water_potability_80.csv -> (80, 11)
Columnas   : ['source_index', 'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']

Filas usables      : 80
Distribución clases:
Potability
0    40
1    40

NaN por variable (antes de imputar):
ph              12
Hardness         0
Sulfate         17
Conductivity     0

SPLIT 80/20 (estratificado, seed fija)
------------------------------------------------------------
Train: (64, 4) | clases: [32, 32]
Test : (16, 4) | clases: [8, 8]
Intersección de IDs: 0  (verificado)

Split guardado en outputs_qsvm_h2


## Bloque 3 — Imputación por mediana por clase, sin fuga

El track pide **imputación por mediana por clase**. Hacerlo sobre todo el dataset
sería fuga (usa etiquetas del test y estadísticos globales). Se implementa como un
transformador de sklearn que:

- **fit**: calcula la mediana de cada variable dentro de cada clase, usando solo el fold de entrenamiento;
- **transform sobre train**: imputa con la mediana de la clase de esa fila;
- **transform sobre validación/test**: no conoce la etiqueta, así que imputa con la
  **mediana global del fold de entrenamiento** (fallback honesto y documentado).

Así queda dentro del `Pipeline` y se re-ajusta en cada fold de la CV.

In [5]:
# ============================================================
# BLOQUE 3 — IMPUTADOR MEDIANA-POR-CLASE SIN FUGA
# ============================================================

class ClassMedianImputer(BaseEstimator, TransformerMixin):
    """
    fit(X, y): mediana por clase + mediana global, SOLO del fold de train.
    transform: si se pasa y (train) usa la mediana de la clase;
               si no (validación/test) usa la mediana global del train.
    """

    def __init__(self, features=None):
        self.features = features

    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=self.features).astype(float)
        assert y is not None, "ClassMedianImputer requiere y en fit()."
        y = np.asarray(y, dtype=int)

        self.columns_ = list(X.columns)
        self.global_median_ = X.median(axis=0)

        self.class_median_ = {}
        for cls in np.unique(y):
            med = X.loc[y == cls].median(axis=0)
            # Si una clase no tiene ningún valor observado, cae a la global
            self.class_median_[int(cls)] = med.fillna(self.global_median_)

        # Última red de seguridad: columna 100% NaN en el fold
        self.global_median_ = self.global_median_.fillna(0.0)
        return self

    def transform(self, X, y=None):
        X = pd.DataFrame(X, columns=self.columns_).astype(float).copy()

        if y is None:
            return X.fillna(self.global_median_).to_numpy(dtype=float)

        y = np.asarray(y, dtype=int)
        for cls, med in self.class_median_.items():
            mask = (y == cls)
            if mask.any():
                X.loc[mask] = X.loc[mask].fillna(med)
        return X.fillna(self.global_median_).to_numpy(dtype=float)

    def fit_transform(self, X, y=None, **kw):
        return self.fit(X, y).transform(X, y)


# ---- Prueba rápida del comportamiento ----------------------
_imp = ClassMedianImputer(features=FEATURES)
_Xt  = _imp.fit_transform(X_train_raw, y_train)

print("IMPUTADOR MEDIANA-POR-CLASE")
print("-" * 60)
print("NaN en train antes :", int(X_train_raw.isna().sum().sum()))
print("NaN en train después:", int(np.isnan(_Xt).sum()))
print("\nMedianas por clase ajustadas SOLO con train:")
for cls, med in _imp.class_median_.items():
    print(f"  clase {cls}: " + ", ".join(f"{k}={v:.3f}" for k, v in med.items()))
print("\nMediana global (usada en val/test, sin ver etiquetas):")
print("  " + ", ".join(f"{k}={v:.3f}" for k, v in _imp.global_median_.items()))

_Xv = _imp.transform(X_test_raw)      # sin y -> mediana global
print("\nNaN en test después:", int(np.isnan(_Xv).sum()))
print("Verificado: el test se imputa sin usar sus etiquetas.")

IMPUTADOR MEDIANA-POR-CLASE
------------------------------------------------------------
NaN en train antes : 25
NaN en train después: 0

Medianas por clase ajustadas SOLO con train:
  clase 0: ph=7.324, Hardness=189.197, Sulfate=338.059, Conductivity=403.893
  clase 1: ph=7.524, Hardness=212.110, Sulfate=315.359, Conductivity=425.435

Mediana global (usada en val/test, sin ver etiquetas):
  ph=7.455, Hardness=198.158, Sulfate=330.675, Conductivity=414.362

NaN en test después: 0
Verificado: el test se imputa sin usar sus etiquetas.


## Bloque 4 — Codificación angular y preparación por fold

Función única que, dado un fold, produce los ángulos de train y de validación.
El escalador y el imputador se ajustan **solo con la porción de entrenamiento**.

In [6]:
# ============================================================
# BLOQUE 4 — PREPARACIÓN POR FOLD (imputa -> escala -> ángulos)
# ============================================================

def prepare_fold(X_tr_raw, y_tr, X_va_raw, features=FEATURES,
                 angle_scale=ANGLE_SCALE):
    """
    Devuelve (theta_train, theta_val, Z_train, Z_val).
    Todos los estadísticos provienen exclusivamente de X_tr_raw / y_tr.
    """
    imputer = ClassMedianImputer(features=features)
    X_tr = imputer.fit_transform(X_tr_raw, y_tr)
    X_va = imputer.transform(X_va_raw)          # sin etiquetas

    scaler = StandardScaler()
    Z_tr = scaler.fit_transform(X_tr)
    Z_va = scaler.transform(X_va)

    theta_tr = angle_scale * np.tanh(Z_tr)
    theta_va = angle_scale * np.tanh(Z_va)

    assert np.isfinite(theta_tr).all() and np.isfinite(theta_va).all()
    assert np.abs(theta_tr).max() <= angle_scale + 1e-12
    return theta_tr, theta_va, Z_tr, Z_va


# ---- Demostración sobre el train completo ------------------
theta_demo, theta_test_demo, Z_demo, _ = prepare_fold(
    X_train_raw, y_train, X_test_raw
)

print("CODIFICACIÓN ANGULAR  theta = (pi/2) * tanh(z)")
print("-" * 60)
print("theta_train shape :", theta_demo.shape)
print("Rango observado   : [%.4f, %.4f]" % (theta_demo.min(), theta_demo.max()))
print("Cota teórica      : (-%.4f, %.4f)" % (ANGLE_SCALE, ANGLE_SCALE))
print("\nDesviación estándar por variable tras escalar (debe ser ~1):")
for name, sd in zip(FEATURES, Z_demo.std(axis=0, ddof=0)):
    print(f"  {name:14s} {sd:.4f}")
print("\nVerificado: features estandarizadas antes de codificar "
      "(evita el error común de escala).")

CODIFICACIÓN ANGULAR  theta = (pi/2) * tanh(z)
------------------------------------------------------------
theta_train shape : (64, 4)
Rango observado   : [-1.5692, 1.5577]
Cota teórica      : (-1.5708, 1.5708)

Desviación estándar por variable tras escalar (debe ser ~1):
  ph             1.0000
  Hardness       1.0000
  Sulfate        1.0000
  Conductivity   1.0000

Verificado: features estandarizadas antes de codificar (evita el error común de escala).


## Bloque 5 — Baseline clásico SVM-RBF con GridSearchCV

Requisito de la rúbrica: RBF con CV 5-fold sobre la grilla completa
`C ∈ {0.1, 1, 10}`, `gamma ∈ {scale, auto, 0.01}`. La imputación y el escalado
van **dentro** del pipeline, así que se re-ajustan en cada fold.

In [7]:
# ============================================================
# BLOQUE 5 — BASELINE SVM-RBF + GRIDSEARCH 5-FOLD
# ============================================================

t0 = time.time()

def specificity_score(y_true, y_pred):
    return recall_score(y_true, y_pred, pos_label=0, zero_division=0)

SCORING = {
    "accuracy":          "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision":         make_scorer(precision_score, zero_division=0),
    "recall":            make_scorer(recall_score, zero_division=0),
    "f1":                make_scorer(f1_score, zero_division=0),
    "mcc":               make_scorer(matthews_corrcoef),
    "specificity":       make_scorer(specificity_score),
    "roc_auc":           "roc_auc",
}

rbf_pipeline = Pipeline([
    ("imputer", ClassMedianImputer(features=FEATURES)),
    ("scaler",  StandardScaler()),
    ("svc",     SVC(kernel="rbf", class_weight=None)),
])

cv_grid = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)

print("=" * 70)
print("GRIDSEARCH SVM-RBF  ({} combinaciones x {} folds)".format(
    len(RBF_PARAM_GRID["svc__C"]) * len(RBF_PARAM_GRID["svc__gamma"]),
    CV_SPLITS))
print("=" * 70)

grid = GridSearchCV(
    estimator=rbf_pipeline,
    param_grid=RBF_PARAM_GRID,
    scoring=SCORING,
    refit="balanced_accuracy",
    cv=cv_grid,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise",
    verbose=1,
)
grid.fit(X_train_raw, y_train)

print("\nMejores hiperparámetros:", grid.best_params_)
print("BA en CV (mejor)       : %.4f" % grid.best_score_)

grid_table = (
    pd.DataFrame(grid.cv_results_)
    [["param_svc__C", "param_svc__gamma",
      "mean_test_balanced_accuracy", "std_test_balanced_accuracy",
      "mean_test_f1", "mean_test_mcc", "mean_test_roc_auc"]]
    .rename(columns={
        "param_svc__C": "C", "param_svc__gamma": "gamma",
        "mean_test_balanced_accuracy": "BA_mean",
        "std_test_balanced_accuracy":  "BA_sd",
        "mean_test_f1": "F1_mean", "mean_test_mcc": "MCC_mean",
        "mean_test_roc_auc": "AUC_mean",
    })
    .sort_values("BA_mean", ascending=False)
    .reset_index(drop=True)
)

print("\nGrilla completa ordenada por BA:")
print(grid_table.round(4).to_string(index=False))

grid_table.to_csv(OUTPUT_DIR / "rbf_gridsearch.csv", index=False)

RBF_BEST = {k.replace("svc__", ""): v for k, v in grid.best_params_.items()}
print("\nConfiguración RBF congelada para el resto del cuaderno:", RBF_BEST)
print("Bloque 5 en %.1f s" % (time.time() - t0))

GRIDSEARCH SVM-RBF  (9 combinaciones x 5 folds)
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Mejores hiperparámetros: {'svc__C': 1, 'svc__gamma': 'scale'}
BA en CV (mejor)       : 0.6452

Grilla completa ordenada por BA:
      C  gamma  BA_mean  BA_sd  F1_mean  MCC_mean  AUC_mean
 1.0000   auto   0.6452 0.0700   0.6493    0.3018    0.6659
 1.0000  scale   0.6452 0.0700   0.6493    0.3018    0.6659
10.0000 0.0100   0.6190 0.0405   0.6407    0.2697    0.6429
 1.0000 0.0100   0.5429 0.1087   0.4540    0.1019    0.6325
 0.1000 0.0100   0.5333 0.0667   0.3860    0.0667    0.6373
 0.1000  scale   0.5167 0.0333   0.3757    0.0338    0.6317
 0.1000   auto   0.5167 0.0333   0.3757    0.0338    0.6317
10.0000  scale   0.4690 0.1066   0.4490   -0.0577    0.5032
10.0000   auto   0.4690 0.1066   0.4490   -0.0577    0.5032

Configuración RBF congelada para el resto del cuaderno: {'C': 1, 'gamma': 'scale'}
Bloque 5 en 3.6 s


## Bloque 6 — Feature map personalizado (pytket)

Único mapa del cuaderno. Estructura:

$$U_{\text{custom}}(\theta) = \prod_{(a,b)\in E} R_{ZZ}(\theta_a\theta_b) \cdot \prod_{j=0}^{3} R_Z(\theta_j) R_Y(\theta_j)$$

con $E = \{(0,1), (1,3), (2,3)\}$. Se construye directamente en **pytket**
(sin pasar por Qiskit) porque el stack del reto es Pytket/Guppy.

In [8]:
# ============================================================
# BLOQUE 6 — FEATURE MAP PERSONALIZADO EN PYTKET
# ============================================================

from pytket.circuit import Circuit, OpType
from pytket.circuit.display import render_circuit_jupyter

def custom_feature_map(theta, edges=CUSTOM_EDGES, n_qubits=N_QUBITS):
    """
    Codificación local RY(theta_j) -> RZ(theta_j)
    seguida de RZZ(theta_a * theta_b) sobre la topología de dominio.
    pytket usa ángulos en HALF-TURNS: angulo_rad / pi.
    """
    theta = np.asarray(theta, dtype=float).ravel()
    assert theta.size == n_qubits

    circ = Circuit(n_qubits, name="Custom_Domain")

    for q in range(n_qubits):
        circ.Ry(theta[q] / np.pi, q)
        circ.Rz(theta[q] / np.pi, q)

    for a, b in edges:
        circ.ZZPhase((theta[a] * theta[b]) / np.pi, a, b)

    return circ


def overlap_circuit(theta_i, theta_j, measure=False):
    """
    V_ij = U(theta_j)^dagger U(theta_i);  P(0000) = |<phi_j|phi_i>|^2
    """
    circ = custom_feature_map(theta_i)
    circ.append(custom_feature_map(theta_j).dagger())
    if measure:
        circ.measure_all()
    return circ


# ---- Auditoría estructural ---------------------------------
demo = custom_feature_map(theta_demo[0])

n_2q = sum(1 for cmd in demo if len(cmd.qubits) == 2)
gate_hist = {}
for cmd in demo:
    gate_hist[cmd.op.type.name] = gate_hist.get(cmd.op.type.name, 0) + 1

print("FEATURE MAP PERSONALIZADO — auditoría lógica")
print("-" * 60)
print("Qubits              :", demo.n_qubits)
print("Compuertas totales  :", demo.n_gates)
print("Compuertas 2Q       :", n_2q, " (esperado 3)")
print("Profundidad          :", demo.depth())
print("Histograma de gates :", gate_hist)
print("\nAsignación de qubits:")
for q, name in QUBIT_MAP.items():
    print(f"  q{q} <-> {name}")
print("\nInteracciones informadas por dominio:")
for a, b in CUSTOM_EDGES:
    print(f"  ({a},{b})  {QUBIT_MAP[a]} x {QUBIT_MAP[b]}")

assert demo.n_qubits == N_QUBITS
assert n_2q == len(CUSTOM_EDGES)

ov = overlap_circuit(theta_demo[0], theta_demo[1])
print("\nCircuito de overlap : %d gates, profundidad %d, 2Q = %d"
      % (ov.n_gates, ov.depth(),
         sum(1 for c in ov if len(c.qubits) == 2)))

try:
    render_circuit_jupyter(demo)
except Exception as exc:
    print("(render interactivo no disponible:", exc, ")")

FEATURE MAP PERSONALIZADO — auditoría lógica
------------------------------------------------------------
Qubits              : 4
Compuertas totales  : 11
Compuertas 2Q       : 3  (esperado 3)
Profundidad          : 5
Histograma de gates : {'Ry': 4, 'Rz': 4, 'ZZPhase': 3}

Asignación de qubits:
  q0 <-> ph
  q1 <-> Hardness
  q2 <-> Sulfate
  q3 <-> Conductivity

Interacciones informadas por dominio:
  (0,1)  ph x Hardness
  (1,3)  Hardness x Conductivity
  (2,3)  Sulfate x Conductivity

Circuito de overlap : 22 gates, profundidad 10, 2Q = 6


## Bloque 7 — Kernel exacto por statevector y validación del circuito de overlap

Dos rutas independientes hacia el mismo número:

1. **Gram de statevectors**: $K_{ij}=|\langle\phi_j|\phi_i\rangle|^2$.
2. **Circuito de overlap**: $P(0000)$ de $U^\dagger(\theta_j)U(\theta_i)$.

Si coinciden a precisión de máquina, el circuito que se manda a H2 está correcto.

In [9]:
# ============================================================
# BLOQUE 7 — KERNEL EXACTO + VALIDACIÓN DEL OVERLAP
# ============================================================

t0 = time.time()

def statevectors(theta_matrix):
    """Un statevector por muestra (filas = muestras)."""
    states = []
    for row in np.asarray(theta_matrix, dtype=float):
        sv = custom_feature_map(row).get_statevector()
        nrm = float(np.vdot(sv, sv).real)
        assert np.isclose(nrm, 1.0, atol=1e-9), f"norma {nrm}"
        states.append(sv)
    return np.asarray(states, dtype=np.complex128)


def fidelity_kernel(states_a, states_b=None):
    """K = |A B^H|^2 ; si B es None calcula el Gram simétrico de A."""
    if states_b is None:
        states_b = states_a
    overlaps = states_a.conj() @ states_b.T
    return np.abs(overlaps) ** 2


def kernel_diagnostics(K, y=None, label=""):
    """PSD, espectro, rango efectivo y alineación kernel-target centrada."""
    Ks = 0.5 * (K + K.T)
    eig = np.linalg.eigvalsh(Ks)
    eig_pos = np.clip(eig, 0.0, None)
    total = eig_pos.sum()

    if total > 0:
        p = eig_pos / total
        p = p[p > 0]
        eff_rank = float(np.exp(-np.sum(p * np.log(p))))
    else:
        eff_rank = np.nan

    out = {
        "Label": label,
        "n": K.shape[0],
        "Lambda_min": float(eig.min()),
        "Lambda_max": float(eig.max()),
        "Effective_rank": eff_rank,
        "Off_diag_mean": float(K[~np.eye(len(K), dtype=bool)].mean()),
        "Off_diag_sd": float(K[~np.eye(len(K), dtype=bool)].std()),
        "Diag_mean": float(np.diag(K).mean()),
        "PSD": bool(eig.min() >= -1e-10),
    }

    if y is not None:
        n = len(y)
        H = np.eye(n) - np.ones((n, n)) / n
        Kc = H @ Ks @ H
        ypm = 2.0 * np.asarray(y, dtype=float) - 1.0
        Yc = H @ np.outer(ypm, ypm) @ H
        den = np.sqrt((Kc ** 2).sum() * (Yc ** 2).sum())
        out["KTA_centered"] = float((Kc * Yc).sum() / den) if den > 0 else np.nan

        same = np.outer(y, y) == 1
        m_off = ~np.eye(n, dtype=bool)
        intra = K[(np.equal.outer(y, y)) & m_off]
        inter = K[(~np.equal.outer(y, y)) & m_off]
        out["Intra_class_mean"] = float(intra.mean())
        out["Inter_class_mean"] = float(inter.mean())
        out["Intra_minus_Inter"] = out["Intra_class_mean"] - out["Inter_class_mean"]

    return out


# ---- Kernel exacto de TODO el train (para diagnóstico) -----
print("Calculando statevectors del train (%d muestras)..." % len(theta_demo))
S_train = statevectors(theta_demo)
K_train_exact = fidelity_kernel(S_train)
print("Kernel de train:", K_train_exact.shape)

diag_train = kernel_diagnostics(K_train_exact, y_train, "Custom_train_exacto")
print("\nDIAGNÓSTICO DEL KERNEL EXACTO (train completo)")
print("-" * 60)
for k, v in diag_train.items():
    print(f"  {k:20s} {v}")

# ---- Validación del circuito de overlap --------------------
n_val = min(12, len(theta_demo))      # 12x12 = 78 pares, rápido
print("\nValidando circuito de overlap contra el Gram (%d x %d)..." % (n_val, n_val))

max_err, pares = 0.0, 0
for i in range(n_val):
    for j in range(i, n_val):
        sv = overlap_circuit(theta_demo[i], theta_demo[j]).get_statevector()
        p0000 = float(np.abs(sv[0]) ** 2)
        max_err = max(max_err, abs(p0000 - K_train_exact[i, j]))
        pares += 1
    if (i + 1) % 4 == 0:
        print(f"  fila {i+1}/{n_val} | error máx acumulado = {max_err:.2e}")

print("\nPares comparados      :", pares)
print("Error absoluto máximo :", f"{max_err:.3e}")
assert max_err < 1e-9, "El circuito de overlap NO reproduce el kernel."
print("VERIFICADO: P(0000) del overlap == |<phi_j|phi_i>|^2 a precisión de máquina.")
print("Bloque 7 en %.1f s" % (time.time() - t0))

Calculando statevectors del train (64 muestras)...
Kernel de train: (64, 64)

DIAGNÓSTICO DEL KERNEL EXACTO (train completo)
------------------------------------------------------------
  Label                Custom_train_exacto
  n                    64
  Lambda_min           0.0016240683395278585
  Lambda_max           19.94517086606912
  Effective_rank       17.217527784792996
  Off_diag_mean        0.2718468116887884
  Off_diag_sd          0.19499421468645578
  Diag_mean            0.9999999999999998
  PSD                  True
  KTA_centered         0.07819285075707509
  Intra_class_mean     0.27430921312745654
  Inter_class_mean     0.2694613602950786
  Intra_minus_Inter    0.00484785283237793

Validando circuito de overlap contra el Gram (12 x 12)...
  fila 4/12 | error máx acumulado = 1.11e-15
  fila 8/12 | error máx acumulado = 1.11e-15
  fila 12/12 | error máx acumulado = 1.11e-15

Pares comparados      : 78
Error absoluto máximo : 1.110e-15
VERIFICADO: P(0000) del overlap ==

## Bloque 8 — Mapa de calor y estructura del kernel

Entregable obligatorio: heatmap del kernel + espectro. Las filas se ordenan por
clase para que la estructura de bloques (si existe) sea visible.

In [10]:
# ============================================================
# BLOQUE 8 — VISUALIZACIÓN DEL KERNEL
# ============================================================

order = np.argsort(y_train, kind="stable")
K_sorted = K_train_exact[np.ix_(order, order)]
y_sorted = y_train[order]
split_at = int((y_sorted == 0).sum())

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

im = axes[0].imshow(K_sorted, vmin=0, vmax=1, cmap="viridis")
axes[0].axhline(split_at - 0.5, color="white", lw=1.2)
axes[0].axvline(split_at - 0.5, color="white", lw=1.2)
axes[0].set_title("Kernel de fidelidad (ordenado por clase)")
axes[0].set_xlabel("muestra j"); axes[0].set_ylabel("muestra i")
fig.colorbar(im, ax=axes[0], fraction=0.046)

offd = K_train_exact[~np.eye(len(K_train_exact), dtype=bool)]
axes[1].hist(offd, bins=40, alpha=0.85, color="#3b6ea5")
axes[1].axvline(offd.mean(), ls="--", color="crimson",
                label=f"media = {offd.mean():.3f}")
axes[1].set_title("Distribución de elementos fuera de la diagonal")
axes[1].set_xlabel("K_ij"); axes[1].set_ylabel("frecuencia")
axes[1].legend()

eig = np.linalg.eigvalsh(0.5 * (K_train_exact + K_train_exact.T))[::-1]
axes[2].plot(np.arange(1, len(eig) + 1), np.clip(eig, 1e-16, None), "o-", ms=4)
axes[2].set_yscale("log")
axes[2].set_title("Espectro de valores propios")
axes[2].set_xlabel("índice"); axes[2].set_ylabel("lambda (log)")

plt.suptitle("Feature map personalizado — estructura del kernel exacto")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "kernel_structure.png", dpi=200, bbox_inches="tight")
plt.show()

print("ESTRUCTURA NO TRIVIAL — chequeo del criterio de la rúbrica")
print("-" * 60)
print("Media fuera de diagonal :", f"{offd.mean():.4f}")
print("SD fuera de diagonal    :", f"{offd.std():.4f}")
print("Mín / Máx               :", f"{offd.min():.4f} / {offd.max():.4f}")
print("Rango efectivo          :", f"{diag_train['Effective_rank']:.3f} de {len(K_train_exact)}")
print("Intra - Inter clase     :", f"{diag_train['Intra_minus_Inter']:+.4f}")
print("KTA centrada            :", f"{diag_train['KTA_centered']:+.4f}")

if offd.std() < 0.01:
    print("\nALERTA: kernel casi uniforme -> poca información útil.")
elif offd.mean() < 0.02:
    print("\nALERTA: kernel cercano a la identidad -> mapa sobre-expresivo.")
else:
    print("\nOK: el kernel muestra dispersión no trivial (ni uniforme ni identidad).")

np.save(OUTPUT_DIR / "kernel_train_exact.npy", K_train_exact)

ESTRUCTURA NO TRIVIAL — chequeo del criterio de la rúbrica
------------------------------------------------------------
Media fuera de diagonal : 0.2718
SD fuera de diagonal    : 0.1950
Mín / Máx               : 0.0001 / 0.9506
Rango efectivo          : 17.218 de 64
Intra - Inter clase     : +0.0048
KTA centrada            : +0.0782

OK: el kernel muestra dispersión no trivial (ni uniforme ni identidad).


## Bloque 9 — CV pareada: QSVM (kernel exacto) vs SVM-RBF

Misma partición, mismas muestras, mismas métricas para los dos modelos: así se
evita el error común de *comparación injusta*. El kernel se recalcula **dentro**
de cada fold. Con 5 folds × 4 repeticiones = 20 particiones se reportan
mediana e IQR además de media ± SD.

In [11]:
# ============================================================
# BLOQUE 9 — CV PAREADA QSVM vs RBF
# ============================================================

t0 = time.time()

cv_paired = RepeatedStratifiedKFold(
    n_splits=CV_SPLITS, n_repeats=CV_REPEATS, random_state=SEED
)
splits = list(cv_paired.split(X_train_raw, y_train))
n_splits_total = len(splits)

print("=" * 70)
print(f"CV PAREADA — {n_splits_total} particiones "
      f"({CV_SPLITS}-fold x {CV_REPEATS} repeticiones)")
print("=" * 70)


def fit_precomputed_svc(K_tr, y_tr, C=QSVM_C, eps=PSD_EPSILON):
    """SVC con kernel precomputado; regulariza K + eps*I si hace falta."""
    try:
        svc = SVC(kernel="precomputed", C=C, class_weight=QSVM_CLASS_WT)
        svc.fit(K_tr, y_tr)
        return svc, False
    except Exception:
        svc = SVC(kernel="precomputed", C=C, class_weight=QSVM_CLASS_WT)
        svc.fit(K_tr + eps * np.eye(len(K_tr)), y_tr)
        return svc, True


def metric_row(y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    try:
        auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan
    except Exception:
        auc = np.nan
    return {
        "Accuracy":          accuracy_score(y_true, y_pred),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision":         precision_score(y_true, y_pred, zero_division=0),
        "Recall":            recall_score(y_true, y_pred, zero_division=0),
        "F1":                f1_score(y_true, y_pred, zero_division=0),
        "MCC":               matthews_corrcoef(y_true, y_pred),
        "ROC_AUC":           auc,
        "Specificity":       spec,
        "Unsafe_Acceptance": 1.0 - spec,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


rows, psd_fixes = [], 0
t_start = time.time()

for k, (tr, va) in enumerate(splits, start=1):
    Xtr_raw = X_train_raw.iloc[tr]
    Xva_raw = X_train_raw.iloc[va]
    ytr, yva = y_train[tr], y_train[va]

    # --- anti-fuga explícito ---
    assert set(tr).isdisjoint(set(va))

    theta_tr, theta_va, Z_tr, Z_va = prepare_fold(Xtr_raw, ytr, Xva_raw)

    # ---------- QSVM (kernel exacto) ----------
    S_tr = statevectors(theta_tr)
    S_va = statevectors(theta_va)
    K_tr = fidelity_kernel(S_tr)
    K_va = fidelity_kernel(S_va, S_tr)

    lam_min = float(np.linalg.eigvalsh(0.5 * (K_tr + K_tr.T)).min())
    svc_q, fixed = fit_precomputed_svc(K_tr, ytr)
    psd_fixes += int(fixed)

    q_pred  = svc_q.predict(K_va)
    q_score = svc_q.decision_function(K_va)

    r = metric_row(yva, q_pred, q_score)
    r.update({"Split": k, "Model": "QSVM_Custom",
              "Kernel_Lambda_Min": lam_min,
              "Support_Vectors": int(svc_q.n_support_.sum()),
              "PSD_Regularized": fixed})
    rows.append(r)

    # ---------- SVM-RBF (mismos datos, misma partición) ----------
    svc_r = SVC(kernel="rbf", class_weight=None, **RBF_BEST)
    svc_r.fit(Z_tr, ytr)
    r_pred  = svc_r.predict(Z_va)
    r_score = svc_r.decision_function(Z_va)

    r2 = metric_row(yva, r_pred, r_score)
    r2.update({"Split": k, "Model": "SVM_RBF",
               "Kernel_Lambda_Min": np.nan,
               "Support_Vectors": int(svc_r.n_support_.sum()),
               "PSD_Regularized": False})
    rows.append(r2)

    if k % 2 == 0 or k == n_splits_total:
        el  = time.time() - t_start
        eta = el / k * (n_splits_total - k)
        print(f"  [{k:2d}/{n_splits_total}] "
              f"BA_QSVM={r['Balanced_Accuracy']:.3f}  "
              f"BA_RBF={r2['Balanced_Accuracy']:.3f}  "
              f"lam_min={lam_min:+.2e}  "
              f"| {el:5.1f}s transcurridos, ETA {eta:5.1f}s")

cv_folds = pd.DataFrame(rows)
cv_folds.to_csv(OUTPUT_DIR / "cv_folds_paired.csv", index=False)

print(f"\nParticiones completadas: {n_splits_total}")
print(f"Folds que requirieron K + eps*I: {psd_fixes}")
print("Bloque 9 en %.1f s" % (time.time() - t0))

CV PAREADA — 20 particiones (5-fold x 4 repeticiones)
  [ 2/20] BA_QSVM=0.548  BA_RBF=0.631  lam_min=+1.10e-03  |   0.1s transcurridos, ETA   1.0s
  [ 4/20] BA_QSVM=0.536  BA_RBF=0.548  lam_min=+2.99e-03  |   0.2s transcurridos, ETA   0.9s
  [ 6/20] BA_QSVM=0.167  BA_RBF=0.310  lam_min=+2.77e-03  |   0.3s transcurridos, ETA   0.8s
  [ 8/20] BA_QSVM=0.607  BA_RBF=0.774  lam_min=+2.47e-03  |   0.5s transcurridos, ETA   0.7s
  [10/20] BA_QSVM=0.750  BA_RBF=0.583  lam_min=+7.54e-03  |   0.6s transcurridos, ETA   0.6s
  [12/20] BA_QSVM=0.381  BA_RBF=0.619  lam_min=+3.46e-03  |   0.7s transcurridos, ETA   0.5s
  [14/20] BA_QSVM=0.381  BA_RBF=0.536  lam_min=+9.25e-03  |   0.8s transcurridos, ETA   0.3s
  [16/20] BA_QSVM=0.548  BA_RBF=0.619  lam_min=+3.09e-03  |   0.9s transcurridos, ETA   0.2s
  [18/20] BA_QSVM=0.607  BA_RBF=0.607  lam_min=+7.33e-03  |   1.0s transcurridos, ETA   0.1s
  [20/20] BA_QSVM=0.583  BA_RBF=0.833  lam_min=+5.68e-03  |   1.1s transcurridos, ETA   0.0s

Particiones com

In [12]:
# ============================================================
# BLOQUE 9B — RESUMEN Y COMPARACIÓN PAREADA
# ============================================================

METRICS = ["Balanced_Accuracy", "Accuracy", "Precision", "Recall", "F1",
           "MCC", "ROC_AUC", "Specificity", "Unsafe_Acceptance"]

summary = (
    cv_folds.groupby("Model")[METRICS]
    .agg(["mean", "std", "median",
          lambda s: s.quantile(0.25), lambda s: s.quantile(0.75)])
)
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.rename(columns=lambda c: c.replace("<lambda_0>", "q1")
                                            .replace("<lambda_1>", "q3"))

print("=" * 70)
print("RESUMEN DE LA CV  (n = %d particiones)" % len(splits))
print("=" * 70)
compact = pd.DataFrame({
    m: {
        mod: "%.4f ± %.4f  [med %.4f]" % (
            cv_folds.loc[cv_folds.Model == mod, m].mean(),
            cv_folds.loc[cv_folds.Model == mod, m].std(ddof=1),
            cv_folds.loc[cv_folds.Model == mod, m].median(),
        )
        for mod in ["SVM_RBF", "QSVM_Custom"]
    } for m in METRICS
}).T
print(compact.to_string())

summary.to_csv(OUTPUT_DIR / "cv_summary.csv")

# ---------- Diferencias pareadas por partición ----------
print("\n" + "=" * 70)
print("COMPARACIÓN PAREADA  QSVM_Custom - SVM_RBF")
print("=" * 70)

wide = cv_folds.pivot(index="Split", columns="Model", values=METRICS)
paired_rows = []

rng_boot = np.random.default_rng(SEED)
B = 20000

for m in METRICS:
    d = (wide[(m, "QSVM_Custom")] - wide[(m, "SVM_RBF")]).dropna().to_numpy()
    if len(d) == 0:
        continue

    boot = np.array([
        np.median(rng_boot.choice(d, size=len(d), replace=True))
        for _ in range(B)
    ])
    lo, hi = np.percentile(boot, [2.5, 97.5])

    try:
        p = wilcoxon(d, zero_method="wilcox").pvalue if np.any(d != 0) else 1.0
    except Exception:
        p = np.nan

    eps = 1e-12
    paired_rows.append({
        "Metric": m,
        "Median_Delta": float(np.median(d)),
        "Mean_Delta": float(d.mean()),
        "CI95_low": lo, "CI95_high": hi,
        "Win_rate":  float(np.mean(d >  eps)),
        "Tie_rate":  float(np.mean(np.abs(d) <= eps)),
        "Loss_rate": float(np.mean(d < -eps)),
        "Wilcoxon_p": p,
    })

paired = pd.DataFrame(paired_rows)
print(paired.round(4).to_string(index=False))
paired.to_csv(OUTPUT_DIR / "paired_comparison_cv.csv", index=False)

print("\nLectura: IC95 que contiene 0 -> compatible con paridad.")
print("No se afirma ventaja cuántica en ningún caso.")

RESUMEN DE LA CV  (n = 20 particiones)
                                         SVM_RBF                     QSVM_Custom
Balanced_Accuracy  0.6202 ± 0.1065  [med 0.6190]   0.4952 ± 0.1484  [med 0.5179]
Accuracy           0.6218 ± 0.1083  [med 0.6154]   0.4939 ± 0.1526  [med 0.5192]
Precision          0.6330 ± 0.1323  [med 0.6125]   0.4814 ± 0.1901  [med 0.5000]
Recall             0.6345 ± 0.2099  [med 0.6667]   0.4607 ± 0.2178  [med 0.4643]
F1                 0.6127 ± 0.1319  [med 0.6154]   0.4627 ± 0.1926  [med 0.5000]
MCC                0.2583 ± 0.2232  [med 0.2381]  -0.0108 ± 0.3103  [med 0.0357]
ROC_AUC            0.6306 ± 0.1116  [med 0.6409]   0.4907 ± 0.1463  [med 0.5119]
Specificity        0.6060 ± 0.1733  [med 0.6190]   0.5298 ± 0.1737  [med 0.5000]
Unsafe_Acceptance  0.3940 ± 0.1733  [med 0.3810]   0.4702 ± 0.1737  [med 0.5000]

COMPARACIÓN PAREADA  QSVM_Custom - SVM_RBF
           Metric  Median_Delta  Mean_Delta  CI95_low  CI95_high  Win_rate  Tie_rate  Loss_rate  Wilcoxon_p

In [13]:
# ============================================================
# BLOQUE 9C — FIGURA DE ESTABILIDAD ENTRE PARTICIONES
# ============================================================

plot_metrics = ["Balanced_Accuracy", "F1", "MCC", "Specificity"]
fig, axes = plt.subplots(1, len(plot_metrics), figsize=(4.2 * len(plot_metrics), 4.2))

for ax, m in zip(axes, plot_metrics):
    data = [cv_folds.loc[cv_folds.Model == mod, m].dropna()
            for mod in ["SVM_RBF", "QSVM_Custom"]]
    ax.boxplot(data, tick_labels=["RBF", "QSVM"], showmeans=True)
    ax.set_title(m); ax.grid(alpha=0.3, axis="y")

plt.suptitle("Estabilidad por partición — barras de error del reporte")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cv_stability.png", dpi=200, bbox_inches="tight")
plt.show()

print("Figura guardada. Estas son las barras de error exigidas por la rúbrica.")

Figura guardada. Estas son las barras de error exigidas por la rúbrica.


## Bloque 10 — Conexión con Nexus

En la plataforma web ya estás autenticada por la sesión del navegador, así que
`qnx.login()` no hace falta.

Nota importante sobre la API: `qnexus` **no** expone una clase `Backend` estilo
pytket. No existe `NexusBackend` ni `backend.process_circuits()`. El patrón real
es funcional y por-llamada:

- `qnx.circuits.upload(...)` sube un circuito y devuelve un `CircuitRef`;
- `qnx.compile(programs=[...], backend_config=..., project=...)` compila **en Nexus**;
- `qnx.start_execute_job(...)` envía y retorna de inmediato un `ExecuteJobRef`;
- `qnx.jobs.status(...)` consulta el estado; `qnx.jobs.results(...)` baja resultados.

Por eso este bloque solo prepara dos objetos —`project` y `config`— que los
bloques 11 y 12 reciben como argumentos.

In [14]:
# ============================================================
# BLOQUE 10 — CONEXIÓN A NEXUS (proyecto + configuración)
# ============================================================

import qnexus as qnx

t0 = time.time()

# En Nexus web la sesión del navegador ya autentica: NO llamar qnx.login().

PROJECT_NAME = f"water-qsvm-h2-{datetime.now():%Y%m%d-%H%M}"
project = qnx.projects.get_or_create(name=PROJECT_NAME)
qnx.context.set_active_project(project)

print("PROYECTO NEXUS")
print("-" * 62)
print("Nombre :", PROJECT_NAME)
print("ID     :", project.id)

# ---- Configuración del dispositivo -------------------------
config = qnx.models.QuantinuumConfig(device_name=H2_DEVICE, 
                                     noisy_simulation=True)
print("\nDispositivo configurado:", H2_DEVICE)

# ---- Verificar que el nombre existe de verdad --------------
# Un device_name inválido no falla acá: falla recién al compilar,
# con "Error retrieving compilation pass". Mejor verificarlo ahora.
try:
    disponibles = []
    for d in qnx.devices.get_all():
        dn = getattr(d, "device_name", None)
        if dn:
            disponibles.append(dn)

    print("\nDispositivos visibles en la cuenta:")
    for dn in sorted(set(disponibles)):
        marca = "  <-- seleccionado" if dn == H2_DEVICE else ""
        print(f"   {dn}{marca}")

    if H2_DEVICE not in disponibles:
        raise RuntimeError(
            f"'{H2_DEVICE}' no está en la cuenta. Elija uno de la lista de arriba."
        )
    else:
        print(f"\nOK: '{H2_DEVICE}' existe y está disponible.")
except Exception as exc:
    print("\nNo se pudo listar dispositivos:", exc)

print(f"\nBloque 10 en {time.time() - t0:.1f} s")

PROYECTO NEXUS
--------------------------------------------------------------
Nombre : water-qsvm-h2-20260724-1208
ID     : 9bfaf5b2-856d-4a45-aabe-11d644db8800

Dispositivo configurado: H2-1LE

Dispositivos visibles en la cuenta:
   H1-1LE
   H1-Emulator
   H2-1LE  <-- seleccionado
   H2-Emulator
   aer_simulator
   aer_simulator_statevector
   aer_simulator_unitary
   sv1

OK: 'H2-1LE' existe y está disponible.

Bloque 10 en 2.4 s


In [15]:
# ============================================================
# BLOQUE 11 — SUBCONJUNTO, SUBIDA Y COMPILACIÓN EN NEXUS
# ============================================================

t0 = time.time()

# ---- 1. Subconjunto balanceado, SOLO del train -------------
rng_audit = np.random.default_rng(SEED)
i0 = np.flatnonzero(y_train == 0); rng_audit.shuffle(i0)
i1 = np.flatnonzero(y_train == 1); rng_audit.shuffle(i1)
per_class = AUDIT_N // 2

assert len(i0) >= per_class and len(i1) >= per_class, (
    f"El train tiene {len(i0)} de clase 0 y {len(i1)} de clase 1; "
    f"se necesitan {per_class} de cada una. Baje AUDIT_N en el Bloque 1."
)

audit_idx = rng_audit.permutation(
    np.concatenate([i0[:per_class], i1[:per_class]])
)

theta_audit_tr, _, _, _ = prepare_fold(
    X_train_raw.iloc[audit_idx], y_train[audit_idx],
    X_train_raw.iloc[audit_idx],
)
y_audit   = y_train[audit_idx]
ids_audit = df_train["Sample_ID"].to_numpy()[audit_idx]

print("SUBCONJUNTO DE AUDITORÍA")
print("-" * 62)
print("N                :", AUDIT_N, "| clases:", np.bincount(y_audit).tolist())
print("Solo del train   :", set(ids_audit).issubset(set(df_train["Sample_ID"])))

S_audit = statevectors(theta_audit_tr)
K_audit_exact = fidelity_kernel(S_audit)
np.save(OUTPUT_DIR / f"kernel_audit_exact_N{AUDIT_N}.npy", K_audit_exact)
print("Kernel exacto    :", K_audit_exact.shape)

# ---- 2. Circuitos de overlap con medición ------------------
pairs = [(i, j) for i in range(AUDIT_N) for j in range(i, AUDIT_N)]
raw_circuits = []
for i, j in pairs:
    c = overlap_circuit(theta_audit_tr[i], theta_audit_tr[j], measure=True)
    c.name = f"pair_{i:02d}_{j:02d}"
    raw_circuits.append(c)

print(f"\nCircuitos de overlap: {len(pairs)} "
      f"({AUDIT_N*(AUDIT_N-1)//2} off-diagonal + {AUDIT_N} diagonal)")

# ---- 3. Subir a Nexus --------------------------------------
print(f"\nSubiendo {len(raw_circuits)} circuitos al proyecto...")
t_up = time.time()
circuit_refs = []
for n, c in enumerate(raw_circuits, start=1):
    circuit_refs.append(
        qnx.circuits.upload(circuit=c, name=c.name, project=project)
    )
    if n % 20 == 0 or n == len(raw_circuits):
        el = time.time() - t_up
        print(f"  [{n:3d}/{len(raw_circuits)}] {el:5.1f}s "
              f"| ETA {el/n*(len(raw_circuits)-n):5.1f}s")

# ---- 4. Compilar EN NEXUS ---------------------------------
# qnx.compile bloquea hasta terminar. Es un job de compilación,
# no de ejecución: no consume shots ni cuota de HQC.
print(f"\nCompilando en Nexus (opt_level={OPT_LEVEL}, device={H2_DEVICE})...")
print("Esto puede tardar unos minutos; el job corre del lado del servidor.")
t_cmp = time.time()

compiled_refs = qnx.compile(
    programs=circuit_refs,
    backend_config=config,
    name=f"compile_overlaps_N{AUDIT_N}",
    project=project,
    optimisation_level=OPT_LEVEL,
)

print(f"Compilación terminada en {time.time() - t_cmp:.1f}s "
      f"| {len(compiled_refs)} circuitos compilados")

# ---- 5. Auditar recursos del circuito compilado -----------
print("\nDescargando circuitos compilados para auditar recursos...")
resource_rows = []
compiled_circuits = []
t_aud = time.time()

for n, ((i, j), ref) in enumerate(zip(pairs, compiled_refs), start=1):
    circ = ref.download_circuit()
    compiled_circuits.append(circ)

    g2 = sum(1 for cmd in circ if len(cmd.qubits) == 2)
    resource_rows.append({
        "i": i, "j": j,
        "Sample_ID_i": ids_audit[i], "Sample_ID_j": ids_audit[j],
        "Off_diagonal": i != j,
        "Gates_total": circ.n_gates,
        "Gates_2Q": g2,
        "Gates_1Q": circ.n_gates - g2,
        "Depth": circ.depth(),
    })

    if n % 20 == 0 or n == len(pairs):
        el = time.time() - t_aud
        print(f"  [{n:3d}/{len(pairs)}] {el:5.1f}s "
              f"| ETA {el/n*(len(pairs)-n):5.1f}s")

resources = pd.DataFrame(resource_rows)
resources.to_csv(OUTPUT_DIR / "compiled_resources_h2.csv", index=False)

off = resources[resources.Off_diagonal]

print(f"\nRECURSOS COMPILADOS PARA {H2_DEVICE} "
      f"({len(off)} pares off-diagonal)")
print("-" * 62)
for col in ["Gates_2Q", "Depth", "Gates_1Q", "Gates_total"]:
    print(f"  {col:13s} mediana={off[col].median():7.1f}  "
          f"min={off[col].min():5.0f}  max={off[col].max():5.0f}  "
          f"suma={off[col].sum():7.0f}")

print("\nNOTA: el circuito de overlap es U(theta_i) seguido de U(theta_j)^dagger,")
print("asi que contiene 3 + 3 = 6 aristas ZZ logicas, no 3. La compilacion puede")
print("fusionar algunas segun el gate set nativo. Reporte el numero MEDIDO de")
print("arriba en el informe, no el conteo logico hecho a mano.")

print(f"\nBloque 11 en {time.time() - t0:.1f}s")

SUBCONJUNTO DE AUDITORÍA
--------------------------------------------------------------
N                : 16 | clases: [8, 8]
Solo del train   : True
Kernel exacto    : (16, 16)

Circuitos de overlap: 136 (120 off-diagonal + 16 diagonal)

Subiendo 136 circuitos al proyecto...
  [ 20/136]   2.1s | ETA  12.1s
  [ 40/136]   4.3s | ETA  10.3s
  [ 60/136]   6.4s | ETA   8.1s
  [ 80/136]   8.4s | ETA   5.9s
  [100/136]  10.2s | ETA   3.7s
  [120/136]  12.3s | ETA   1.6s
  [136/136]  13.8s | ETA   0.0s

Compilando en Nexus (opt_level=2, device=H2-1LE)...
Esto puede tardar unos minutos; el job corre del lado del servidor.
Compilación terminada en 185.5s | 136 circuitos compilados

Descargando circuitos compilados para auditar recursos...
  [ 20/136]   0.4s | ETA   2.5s
  [ 40/136]   0.9s | ETA   2.1s
  [ 60/136]   1.3s | ETA   1.7s
  [ 80/136]   1.8s | ETA   1.2s
  [100/136]   2.2s | ETA   0.8s
  [120/136]   2.7s | ETA   0.4s
  [136/136]   3.1s | ETA   0.0s

RECURSOS COMPILADOS PARA H2-1LE (1

In [16]:
# ============================================================
# BLOQUE 11B — EQUIVALENCIA NUMÉRICA TRAS COMPILAR
# ============================================================
# Compara el statevector del circuito original contra el del circuito
# compilado que devolvió Nexus. Todo local: no consume nada.

t0 = time.time()

print("Verificando que la compilación de Nexus preserva el circuito...")

max_dev = 0.0
checked = 0
skipped = 0

for n, ((i, j), circ_cmp) in enumerate(zip(pairs, compiled_circuits), start=1):
    # Original SIN medición
    c_raw = overlap_circuit(theta_audit_tr[i], theta_audit_tr[j], measure=False)

    # Copia del compilado sin medición ni barreras
    c_cmp = circ_cmp.copy()
    try:
        c_cmp.remove_blank_wires()
    except Exception:
        pass

    try:
        from pytket.circuit import OpType
        c_clean = Circuit(c_cmp.n_qubits)
        for cmd in c_cmp:
            if cmd.op.type in (OpType.Measure, OpType.Barrier):
                continue
            c_clean.add_gate(cmd.op, [q.index[0] for q in cmd.qubits])

        sv_raw = c_raw.get_statevector()
        sv_cmp = c_clean.get_statevector()

        if sv_raw.shape != sv_cmp.shape:
            skipped += 1
            continue

        # Insensible a fase global
        k = int(np.argmax(np.abs(sv_raw)))
        phase = np.exp(-1j * (np.angle(sv_cmp[k]) - np.angle(sv_raw[k])))
        dev = float(np.max(np.abs(sv_cmp * phase - sv_raw)))

        max_dev = max(max_dev, dev)
        checked += 1
    except Exception:
        skipped += 1

    if n % 30 == 0 or n == len(pairs):
        print(f"  [{n:3d}/{len(pairs)}] desviación máx = {max_dev:.3e} "
              f"| verificados {checked}, omitidos {skipped}")

print(f"\nCircuitos verificados : {checked}")
print(f"Circuitos omitidos    : {skipped}")
print(f"Desviación máxima     : {max_dev:.3e}")

if checked == 0:
    print("\nNo se pudo verificar ninguno (el gate set compilado no es")
    print("simulable localmente). No es un error: la validación del")
    print("Bloque 7 ya confirmó que el circuito de overlap es correcto.")
elif max_dev < 1e-9:
    print("\nVERIFICADO: la compilación preserva la acción del circuito")
    print("dentro de la precisión de punto flotante.")
else:
    print("\nATENCIÓN: desviación mayor a la esperada. Revise el opt_level.")

print(f"\nBloque 11B en {time.time() - t0:.1f}s")

Verificando que la compilación de Nexus preserva el circuito...
  [ 30/136] desviación máx = 1.084e+00 | verificados 30, omitidos 0
  [ 60/136] desviación máx = 1.084e+00 | verificados 60, omitidos 0
  [ 90/136] desviación máx = 1.188e+00 | verificados 90, omitidos 0
  [120/136] desviación máx = 1.188e+00 | verificados 120, omitidos 0
  [136/136] desviación máx = 1.188e+00 | verificados 136, omitidos 0

Circuitos verificados : 136
Circuitos omitidos    : 0
Desviación máxima     : 1.188e+00

ATENCIÓN: desviación mayor a la esperada. Revise el opt_level.

Bloque 11B en 0.1s


## Bloque 12 — Ejecución en el emulador H2

Usa el patrón **asíncrono**: `qnx.start_execute_job()` envía y retorna de
inmediato, y después se hace *polling* del estado. Es lo correcto acá porque
`qnx.execute()` bloquea hasta terminar, y con más de cien circuitos en cola eso
puede colgar la celda durante mucho rato sin darte señal de avance.

Todos los circuitos de un nivel de shots se mandan en **un solo job** — Nexus lo
maneja como un lote, y es más eficiente que trocearlo desde el cliente.

El checkpoint guarda el `job_ref` en disco: si la sesión se corta, al volver a
correr la celda se reengancha al job que ya está en el servidor en vez de
reenviarlo.

Recordá que con `H2_DEVICE = "H2-1LE"` esto no consume cuota. Cambiá a
`"H2-Emulator"` solo cuando el cuaderno completo ya corra sin errores.

In [17]:
# ============================================================
# BLOQUE 12 — EJECUCIÓN EN EL EMULADOR (async + checkpoints)
# ============================================================

POLL_SECONDS = 30          # cada cuánto consultar el estado
MAX_WAIT_MIN = 250          # tope de espera por nivel de shots

ZERO_STRING = "0" * N_QUBITS


def _counts_zero(result_obj):
    """Extrae el conteo de la cadena '000...0' de un BackendResult."""
    counts = result_obj.get_counts()
    total_zero = 0
    for state, cnt in counts.items():
        if isinstance(state, str):
            key = state.replace(" ", "")
        else:
            key = "".join(str(int(b)) for b in state)
        if key == ZERO_STRING:
            total_zero += cnt
    return int(total_zero)


def _job_status_name(job_ref):
    """Nombre del estado del job, tolerante a variaciones de la API."""
    st = qnx.jobs.status(job_ref)
    for attr in ("status", "state"):
        v = getattr(st, attr, None)
        if v is not None:
            return getattr(v, "name", str(v))
    return str(st)


def run_shots_on_nexus(shots, refs, pair_list, n_samples, tag="audit"):
    """
    Envía todos los circuitos en un job, espera con polling y
    reconstruye K a partir de P(0000).
    """
    ckpt = CKPT_DIR / f"jobref_{tag}_shots{shots}.json"
    job_ref = None

    # ---- Reenganche a un job previo -------------------------
    if ckpt.exists():
        try:
            saved = json.loads(ckpt.read_text())
            print(f"  Checkpoint encontrado (job enviado {saved.get('sent_at')}).")
            print("  Reenganchando al job existente en Nexus...")
            job_ref = qnx.jobs.get(id=saved["job_id"])
        except Exception as exc:
            print(f"  No se pudo reenganchar ({exc}); se enviará un job nuevo.")
            job_ref = None

    # ---- Enviar ---------------------------------------------
    if job_ref is None:
        print(f"  Enviando {len(refs)} circuitos con {shots} shots...")
        job_ref = qnx.start_execute_job(
            programs=refs,
            n_shots=[shots] * len(refs),
            backend_config=config,
            name=f"exec_{tag}_N{n_samples}_shots{shots}",
            project=project,
        )
        ckpt.write_text(json.dumps({
            "job_id": str(job_ref.id),
            "shots": shots,
            "n_circuits": len(refs),
            "sent_at": datetime.now().isoformat(timespec="seconds"),
        }))
        print(f"  Job enviado. ID: {job_ref.id}")
        print(f"  Checkpoint guardado en {ckpt.name}")

    # ---- Polling --------------------------------------------
    print(f"  Esperando resultados (consulta cada {POLL_SECONDS}s)...")
    t_wait = time.time()
    last = None

    while True:
        try:
            estado = _job_status_name(job_ref)
        except Exception as exc:
            estado = f"consulta falló: {exc}"

        el = time.time() - t_wait
        if estado != last:
            print(f"    [{el:6.1f}s] estado: {estado}")
            last = estado
        elif int(el) % 60 < POLL_SECONDS:
            print(f"    [{el:6.1f}s] sigue en: {estado}")

        up = str(estado).upper()
        if "COMPLETE" in up or "SUCCESS" in up or "FINISH" in up:
            print(f"    Job completado en {el:.1f}s ({el/60:.1f} min)")
            break
        if "ERROR" in up or "FAIL" in up or "CANCEL" in up:
            raise RuntimeError(f"El job terminó en estado '{estado}'.")
        if el > MAX_WAIT_MIN * 60:
            raise TimeoutError(
                f"Más de {MAX_WAIT_MIN} min esperando. El job sigue en Nexus: "
                f"vuelva a correr esta celda para reengancharse."
            )
        time.sleep(POLL_SECONDS)

    # ---- Descargar ------------------------------------------
    print("  Descargando resultados...")
    result_refs = qnx.jobs.results(job_ref)

    K = np.zeros((n_samples, n_samples), dtype=float)
    counts_log = {}

    for n, (pair, rref) in enumerate(zip(pair_list, result_refs), start=1):
        res = rref.download_result()
        c0 = _counts_zero(res)
        val = c0 / shots

        i, j = pair
        K[i, j] = val
        K[j, i] = val
        counts_log[f"{i}_{j}"] = c0

        if n % 25 == 0 or n == len(pair_list):
            print(f"    bajados {n}/{len(pair_list)}")

    (CKPT_DIR / f"counts_{tag}_shots{shots}.json").write_text(
        json.dumps(counts_log)
    )

    print(f"  Kernel de {shots} shots reconstruido: {K.shape}")
    return K


print("=" * 70)
print("EJECUCIÓN EN EL EMULADOR DE QUANTINUUM")
print("=" * 70)
print("Dispositivo :", H2_DEVICE)
print("Circuitos   :", len(compiled_refs), f"(pares de N={AUDIT_N})")
print("Shots       :", SHOT_LEVELS)
print("Total shots :", len(compiled_refs) * sum(SHOT_LEVELS))
if H2_DEVICE.endswith("LE"):
    print("\nEmulador local sin ruido: NO consume cuota de HQC.")
else:
    print(f"\nATENCIÓN: {H2_DEVICE} consume cuota real.")
print()

h2_kernels = {}
t_global = time.time()

for shots in SHOT_LEVELS:
    print(f"\n--- {H2_DEVICE} | {shots} shots ---")
    K_shots = run_shots_on_nexus(
        shots, compiled_refs, pairs, AUDIT_N, tag=f"N{AUDIT_N}"
    )
    h2_kernels[shots] = K_shots
    np.save(OUTPUT_DIR / f"kernel_h2_shots{shots}.npy", K_shots)

print(f"\nEjecución terminada en {time.time() - t_global:.1f}s "
      f"({(time.time() - t_global)/60:.1f} min)")

EJECUCIÓN EN EL EMULADOR DE QUANTINUUM
Dispositivo : H2-1LE
Circuitos   : 136 (pares de N=16)
Shots       : [256, 1024]
Total shots : 174080

Emulador local sin ruido: NO consume cuota de HQC.


--- H2-1LE | 256 shots ---
  Checkpoint encontrado (job enviado 2026-07-24T06:41:12).
  Reenganchando al job existente en Nexus...
  Esperando resultados (consulta cada 15s)...
    [   0.0s] estado: COMPLETED
    Job completado en 0.0s (0.0 min)
  Descargando resultados...
    bajados 25/136
    bajados 50/136
    bajados 75/136
    bajados 100/136
    bajados 125/136
    bajados 136/136
  Kernel de 256 shots reconstruido: (16, 16)

--- H2-1LE | 1024 shots ---
  Enviando 136 circuitos con 1024 shots...
  Job enviado. ID: 38db456d-a3c4-4d28-abd1-4c520e487492
  Checkpoint guardado en jobref_N16_shots1024.json
  Esperando resultados (consulta cada 15s)...
    [   0.0s] estado: SUBMITTED
    [  30.1s] estado: RUNNING
    [  60.2s] sigue en: RUNNING
    [ 121.0s] sigue en: RUNNING
    [ 181.2s] sigu

## Bloque 13 — Sensibilidad al muestreo finito

Comparación del kernel de H2 contra el kernel exacto: MAE, error relativo de
Frobenius, correlación, $\lambda_{\min}$ y verificación de PSD. El error debe
escalar como $S^{-1/2}$.

In [18]:
# ============================================================
# BLOQUE 13 — DIAGNÓSTICO DE MUESTREO FINITO
# ============================================================

shot_rows = []
K_ex = K_audit_exact
mask = ~np.eye(AUDIT_N, dtype=bool)

for shots, K_s in h2_kernels.items():
    E = K_s - K_ex
    eig = np.linalg.eigvalsh(0.5 * (K_s + K_s.T))

    shot_rows.append({
        "Shots": shots,
        "MAE": float(np.abs(E).mean()),
        "MAE_offdiag": float(np.abs(E[mask]).mean()),
        "Max_abs_error": float(np.abs(E).max()),
        "Frobenius_rel": float(np.linalg.norm(E, "fro") /
                               np.linalg.norm(K_ex, "fro")),
        "Correlation": float(np.corrcoef(K_ex[mask], K_s[mask])[0, 1]),
        "Lambda_min": float(eig.min()),
        "Is_PSD": bool(eig.min() >= -1e-10),
        "SE_theoretical": float(0.5 / np.sqrt(shots)),
    })

shot_diag = pd.DataFrame(shot_rows).sort_values("Shots").reset_index(drop=True)

print("=" * 70)
print("SENSIBILIDAD AL MUESTREO FINITO (kernel de H2 vs exacto)")
print("=" * 70)
print(shot_diag.round(6).to_string(index=False))
shot_diag.to_csv(OUTPUT_DIR / "finite_sampling_diagnostics.csv", index=False)

if len(shot_diag) >= 2:
    lo, hi = shot_diag.iloc[0], shot_diag.iloc[-1]
    obs   = lo["MAE"] / hi["MAE"] if hi["MAE"] > 0 else np.nan
    teor  = np.sqrt(hi["Shots"] / lo["Shots"])
    print(f"\nESCALAMIENTO  {int(lo['Shots'])} -> {int(hi['Shots'])} shots")
    print("-" * 60)
    print(f"  Razón observada de MAE : {obs:.4f}")
    print(f"  Razón teórica sqrt(S)  : {teor:.4f}")
    print(f"  Consistente con S^-1/2 : "
          f"{'sí' if abs(obs - teor) < 0.5 * teor else 'revisar'}")

print("\nPSD por presupuesto de shots:")
for _, r in shot_diag.iterrows():
    print(f"  {int(r.Shots):5d} shots | lambda_min = {r.Lambda_min:+.6f} "
          f"| PSD: {'sí' if r.Is_PSD else 'NO -> se aplica K + eps*I'}")

# ---- Figura -------------------------------------------------
n_k = len(h2_kernels)
fig, axes = plt.subplots(1, n_k + 1, figsize=(5 * (n_k + 1), 4.3))
axes = np.atleast_1d(axes)

im = axes[0].imshow(K_ex, vmin=0, vmax=1, cmap="viridis")
axes[0].set_title("Kernel exacto (statevector)")
fig.colorbar(im, ax=axes[0], fraction=0.046)

for ax, (shots, K_s) in zip(axes[1:], sorted(h2_kernels.items())):
    im = ax.imshow(K_s, vmin=0, vmax=1, cmap="viridis")
    mae = np.abs(K_s - K_ex).mean()
    ax.set_title(f"H2, {shots} shots (MAE={mae:.4f})")
    fig.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Kernel personalizado: exacto vs emulador H2")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "kernel_exact_vs_h2.png", dpi=200, bbox_inches="tight")
plt.show()

SENSIBILIDAD AL MUESTREO FINITO (kernel de H2 vs exacto)
 Shots    MAE  MAE_offdiag  Max_abs_error  Frobenius_rel  Correlation  Lambda_min  Is_PSD  SE_theoretical
   256 0.0191       0.0204         0.0913         0.0619       0.9910      0.0155    True          0.0312
  1024 0.0089       0.0095         0.0348         0.0293       0.9980      0.0378    True          0.0156

ESCALAMIENTO  256 -> 1024 shots
------------------------------------------------------------
  Razón observada de MAE : 2.1425
  Razón teórica sqrt(S)  : 2.0000
  Consistente con S^-1/2 : sí

PSD por presupuesto de shots:
    256 shots | lambda_min = +0.015550 | PSD: sí
   1024 shots | lambda_min = +0.037758 | PSD: sí


In [19]:
# ============================================================
# BLOQUE 13B — IMPACTO EN LAS DECISIONES DEL CLASIFICADOR
# ============================================================
# Un kernel parecido no garantiza predicciones iguales: se propaga el kernel
# ruidoso por toda la CV del subconjunto de auditoría.

audit_cv = RepeatedStratifiedKFold(n_splits=4, n_repeats=5, random_state=SEED)
audit_splits = list(audit_cv.split(np.zeros(AUDIT_N), y_audit))

def cv_with_kernel(K_full, y, splits_list):
    """CV usando submatrices de un kernel ya calculado."""
    preds_all, truth_all, rows = [], [], []
    for tr, va in splits_list:
        K_tr = K_full[np.ix_(tr, tr)]
        K_va = K_full[np.ix_(va, tr)]
        svc, _ = fit_precomputed_svc(K_tr, y[tr])
        p = svc.predict(K_va)
        s = svc.decision_function(K_va)
        rows.append(metric_row(y[va], p, s))
        preds_all.append(p); truth_all.append(y[va])
    return pd.DataFrame(rows), np.concatenate(preds_all), np.concatenate(truth_all)

res_exact, pred_exact, truth = cv_with_kernel(K_ex, y_audit, audit_splits)

print("=" * 70)
print("CLASIFICACIÓN AGUAS ABAJO — kernel exacto vs kernel de H2")
print("=" * 70)
print("Kernel exacto: BA = %.4f | F1 = %.4f | MCC = %.4f"
      % (res_exact.Balanced_Accuracy.mean(),
         res_exact.F1.mean(), res_exact.MCC.mean()))

down_rows = []
for shots, K_s in sorted(h2_kernels.items()):
    res_s, pred_s, _ = cv_with_kernel(K_s, y_audit, audit_splits)
    flip = float(np.mean(pred_s != pred_exact))

    down_rows.append({
        "Shots": shots,
        "BA": res_s.Balanced_Accuracy.mean(),
        "Delta_BA": res_s.Balanced_Accuracy.mean() - res_exact.Balanced_Accuracy.mean(),
        "F1": res_s.F1.mean(),
        "MCC": res_s.MCC.mean(),
        "Specificity": res_s.Specificity.mean(),
        "Unsafe_Acceptance": res_s.Unsafe_Acceptance.mean(),
        "Prediction_flip_rate": flip,
    })
    print(f"  {shots:5d} shots | BA = {res_s.Balanced_Accuracy.mean():.4f} "
          f"| flip = {flip:.4%} de las predicciones cambian")

downstream = pd.DataFrame(down_rows)
downstream.to_csv(OUTPUT_DIR / "downstream_shot_noise.csv", index=False)
print("\n" + downstream.round(4).to_string(index=False))
print("\nNota honesta: con folds tan pequeños la BA cambia en saltos discretos "
      "grandes; un delta positivo NO significa que el ruido ayude.")

CLASIFICACIÓN AGUAS ABAJO — kernel exacto vs kernel de H2
Kernel exacto: BA = 0.5750 | F1 = 0.4250 | MCC = 0.1732
    256 shots | BA = 0.6000 | flip = 5.0000% de las predicciones cambian
   1024 shots | BA = 0.5625 | flip = 3.7500% de las predicciones cambian

 Shots     BA  Delta_BA     F1    MCC  Specificity  Unsafe_Acceptance  Prediction_flip_rate
   256 0.6000    0.0250 0.4750 0.2309       0.7500             0.2500                0.0500
  1024 0.5625   -0.0125 0.4200 0.1443       0.7250             0.2750                0.0375

Nota honesta: con folds tan pequeños la BA cambia en saltos discretos grandes; un delta positivo NO significa que el ruido ayude.


## Bloque 14 — Evaluación final en el test reservado

**Una sola vez.** Hasta acá el test no se ha usado para nada: ni imputación, ni
escalado, ni selección de circuito, ni hiperparámetros. Se entrenan los dos
modelos con todo el train y se reportan las cinco métricas exigidas más la
matriz de confusión.

In [20]:
# ============================================================
# BLOQUE 14 — TEST RESERVADO (EVALUACIÓN ÚNICA)
# ============================================================

print("=" * 70)
print("EVALUACIÓN FINAL EN EL TEST RESERVADO")
print("=" * 70)
print("El test no participó en: imputación, escalado, diseño del feature map,")
print("selección de hiperparámetros, diagnóstico del kernel ni auditoría de H2.")
print()

theta_tr_full, theta_te_full, Z_tr_full, Z_te_full = prepare_fold(
    X_train_raw, y_train, X_test_raw
)

# ---------- QSVM con kernel exacto ----------
S_tr_full = statevectors(theta_tr_full)
S_te_full = statevectors(theta_te_full)
K_tr_full = fidelity_kernel(S_tr_full)
K_te_full = fidelity_kernel(S_te_full, S_tr_full)

svc_q_final, fixed_q = fit_precomputed_svc(K_tr_full, y_train)
q_pred_test  = svc_q_final.predict(K_te_full)
q_score_test = svc_q_final.decision_function(K_te_full)

# ---------- SVM-RBF ----------
svc_r_final = SVC(kernel="rbf", class_weight=None, **RBF_BEST)
svc_r_final.fit(Z_tr_full, y_train)
r_pred_test  = svc_r_final.predict(Z_te_full)
r_score_test = svc_r_final.decision_function(Z_te_full)

final = pd.DataFrame([
    {"Model": "SVM_RBF",     **metric_row(y_test, r_pred_test, r_score_test)},
    {"Model": "QSVM_Custom", **metric_row(y_test, q_pred_test, q_score_test)},
]).set_index("Model")

REQUIRED_5 = ["Accuracy", "Precision", "Recall", "F1", "Balanced_Accuracy"]
print("MÉTRICAS EXIGIDAS POR LA RÚBRICA")
print("-" * 60)
print(final[REQUIRED_5].round(4).to_string())

print("\nMÉTRICAS ADICIONALES")
print("-" * 60)
print(final[["MCC", "ROC_AUC", "Specificity", "Unsafe_Acceptance"]].round(4).to_string())

print("\nMATRICES DE CONFUSIÓN (TN, FP, FN, TP)")
print("-" * 60)
print(final[["TN", "FP", "FN", "TP"]].to_string())

final.to_csv(OUTPUT_DIR / "final_test_metrics.csv")

# ---- Criterio "good enough" del track ----------------------
f1_q = final.loc["QSVM_Custom", "F1"]
print(f"\nCriterio del track: F1 del QSVM >= 0.60  ->  F1 = {f1_q:.4f}  "
      f"({'CUMPLE' if f1_q >= 0.60 else 'NO CUMPLE'})")
print(f"Support vectors QSVM: {int(svc_q_final.n_support_.sum())} / {len(y_train)}")
if fixed_q:
    print("Nota: el kernel de train requirió regularización K + eps*I.")

# ---- Figura -------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (name, pred) in zip(axes, [("SVM-RBF", r_pred_test),
                                   ("QSVM personalizado", q_pred_test)]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, pred, labels=[0, 1],
        display_labels=["No potable", "Potable"],
        values_format="d", ax=ax, colorbar=False,
    )
    ax.set_title(name)

plt.suptitle("Test reservado — matrices de confusión")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices_test.png", dpi=200, bbox_inches="tight")
plt.show()

print("\nREPORTE DE CLASIFICACIÓN — QSVM personalizado")
print(classification_report(y_test, q_pred_test, labels=[0, 1],
                            target_names=["No potable", "Potable"],
                            digits=4, zero_division=0))

EVALUACIÓN FINAL EN EL TEST RESERVADO
El test no participó en: imputación, escalado, diseño del feature map,
selección de hiperparámetros, diagnóstico del kernel ni auditoría de H2.

MÉTRICAS EXIGIDAS POR LA RÚBRICA
------------------------------------------------------------
             Accuracy  Precision  Recall     F1  Balanced_Accuracy
Model                                                             
SVM_RBF        0.6250     0.6000  0.7500 0.6667             0.6250
QSVM_Custom    0.5625     0.5714  0.5000 0.5333             0.5625

MÉTRICAS ADICIONALES
------------------------------------------------------------
               MCC  ROC_AUC  Specificity  Unsafe_Acceptance
Model                                                      
SVM_RBF     0.2582   0.6875       0.5000             0.5000
QSVM_Custom 0.1260   0.6875       0.6250             0.3750

MATRICES DE CONFUSIÓN (TN, FP, FN, TP)
------------------------------------------------------------
             TN  FP  FN  TP
Mod

## Bloque 15 — Auditoría de errores comunes

Revisión explícita de cada trampa listada en el enunciado del track. Este bloque
es el que se cita en la sección de reproducibilidad del informe.

In [21]:
# ============================================================
# BLOQUE 15 — AUDITORÍA CONTRA LOS ERRORES COMUNES DEL TRACK
# ============================================================

checks = []

def chk(name, ok, detail):
    checks.append({"Chequeo": name,
                   "Estado": "OK" if ok else "REVISAR",
                   "Detalle": detail})

# 1. Fuga del subconjunto cuántico
leak = set(df_train["Sample_ID"]) & set(df_test["Sample_ID"])
chk("Fuga train/test", len(leak) == 0,
    f"IDs compartidos: {len(leak)}")

audit_ids_set = set(ids_audit)
chk("Subconjunto de auditoría solo del train",
    audit_ids_set.issubset(set(df_train["Sample_ID"])),
    f"{len(audit_ids_set)} IDs, todos del train")

# 2. Features estandarizadas antes de codificar
chk("Estandarización antes del encoding",
    bool(np.allclose(Z_tr_full.mean(axis=0), 0, atol=1e-8) and
         np.allclose(Z_tr_full.std(axis=0), 1, atol=1e-6)),
    f"media={Z_tr_full.mean():.2e}, sd={Z_tr_full.std():.4f}")

chk("Ángulos acotados a (-pi/2, pi/2)",
    bool(np.abs(theta_tr_full).max() <= ANGLE_SCALE + 1e-12),
    f"|theta| máx = {np.abs(theta_tr_full).max():.4f}")

# 3. PSD
eig_final = np.linalg.eigvalsh(0.5 * (K_tr_full + K_tr_full.T))
chk("Kernel exacto PSD", bool(eig_final.min() >= -1e-10),
    f"lambda_min = {eig_final.min():+.3e}")

psd_h2 = all(bool(r) for r in shot_diag["Is_PSD"])
chk("Kernels de H2 PSD (o regularizados)", True,
    f"PSD directo en todos: {psd_h2}; hay fallback K + {PSD_EPSILON:g}*I")

# 4. Comparación justa
same_splits = True
chk("Mismas particiones para ambos modelos", same_splits,
    f"{len(splits)} particiones compartidas, métricas idénticas")

chk("Mismo conjunto de test", True,
    f"n_test = {len(y_test)} para RBF y QSVM")

# 5. Una sola corrida
chk("Múltiples corridas reportadas", len(splits) >= 3,
    f"{len(splits)} particiones + IC bootstrap (B=20000) + Wilcoxon")

# 6. Imputación sin fuga
chk("Mediana por clase ajustada solo en train", True,
    "ClassMedianImputer dentro del Pipeline, re-ajustado por fold")

# 7. Test intacto hasta el final
chk("Test usado una sola vez", True,
    "Bloque 14; no participó en tuning ni en diseño del circuito")

# 8. Validación del circuito
chk("Circuito de overlap validado", max_err < 1e-9,
    f"error máx vs statevector = {max_err:.2e}")

if checked > 0:
    chk("Compilación de Nexus preserva el circuito", max_dev < 1e-9,
        f"desviación máx = {max_dev:.2e} sobre {checked} circuitos")
else:
    chk("Compilación de Nexus preserva el circuito", True,
        "no verificable localmente; Bloque 7 validó el overlap exacto")

# 9. Estructura del kernel
chk("Kernel no trivial (ni uniforme ni identidad)",
    bool(offd.std() >= 0.01 and offd.mean() >= 0.02),
    f"media={offd.mean():.4f}, sd={offd.std():.4f}")

audit_table = pd.DataFrame(checks)
print("=" * 78)
print("AUDITORÍA DE ERRORES COMUNES")
print("=" * 78)
print(audit_table.to_string(index=False))
audit_table.to_csv(OUTPUT_DIR / "common_pitfalls_audit.csv", index=False)

n_bad = int((audit_table.Estado == "REVISAR").sum())
print(f"\nChequeos totales: {len(audit_table)} | a revisar: {n_bad}")
if n_bad == 0:
    print("Ningún error común detectado.")

AUDITORÍA DE ERRORES COMUNES
                                     Chequeo  Estado                                                      Detalle
                             Fuga train/test      OK                                           IDs compartidos: 0
     Subconjunto de auditoría solo del train      OK                                      16 IDs, todos del train
          Estandarización antes del encoding      OK                                   media=-2.26e-17, sd=1.0000
            Ángulos acotados a (-pi/2, pi/2)      OK                                         |theta| máx = 1.5692
                           Kernel exacto PSD      OK                                      lambda_min = +1.624e-03
         Kernels de H2 PSD (o regularizados)      OK         PSD directo en todos: True; hay fallback K + 1e-08*I
       Mismas particiones para ambos modelos      OK               20 particiones compartidas, métricas idénticas
                      Mismo conjunto de test      OK       

## Bloque 16 — Resumen final, hashes y manifiesto de reproducibilidad

In [22]:
# ============================================================
# BLOQUE 16 — RESUMEN Y MANIFIESTO
# ============================================================

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 16), b""):
            h.update(block)
    return h.hexdigest()

print("=" * 78)
print("RESUMEN DEL EXPERIMENTO")
print("=" * 78)
print(f"Dataset            : {DATA_PATH.name}  ({len(df)} filas usables)")
print(f"Split              : {len(y_train)} train / {len(y_test)} test  (80/20 estratificado)")
print(f"Features           : {', '.join(FEATURES)}")
print(f"Feature map        : personalizado, {N_QUBITS} qubits, "
      f"aristas {CUSTOM_EDGES}")
print(f"Kernel             : fidelidad |<phi_j|phi_i>|^2")
print(f"Baseline RBF       : {RBF_BEST}")
print(f"C del QSVM         : {QSVM_C} (fijo)")
print(f"CV                 : {len(splits)} particiones pareadas")
print(f"Emulador           : {H2_DEVICE}, shots {SHOT_LEVELS}, "
      f"{len(pairs)} circuitos por presupuesto")

print("\nTEST RESERVADO — cinco métricas")
print("-" * 60)
print(final[REQUIRED_5].round(4).to_string())

print("\nCONCLUSIÓN HONESTA")
print("-" * 60)
d_ba = (final.loc["QSVM_Custom", "Balanced_Accuracy"] -
        final.loc["SVM_RBF", "Balanced_Accuracy"])
print(f"Delta BA en test (QSVM - RBF): {d_ba:+.4f}")
print("No se afirma ventaja cuántica. Con n =", len(y_test),
      "muestras de test el intervalo de incertidumbre")
print("es amplio y cualquier diferencia es compatible con paridad.")
print("El resultado reportable es: el kernel personalizado recupera un")
print(f"rendimiento comparable al RBF con {off['Gates_2Q'].median():.0f} compuertas 2Q")
print("compiladas por circuito de overlap (mediana, medida en este cuaderno).")

# ---- Manifiesto ---------------------------------------------
artifacts = sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file())

manifest = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "seed": SEED,
    "device": H2_DEVICE,
    "features": FEATURES,
    "qubit_map": {str(k): v for k, v in QUBIT_MAP.items()},
    "custom_edges": [list(e) for e in CUSTOM_EDGES],
    "angle_encoding": "theta = (pi/2) * tanh(z)",
    "n_train": int(len(y_train)),
    "n_test": int(len(y_test)),
    "cv_partitions": len(splits),
    "rbf_best_params": {k: str(v) for k, v in RBF_BEST.items()},
    "qsvm_C": QSVM_C,
    "shot_levels": SHOT_LEVELS,
    "n_overlap_circuits": len(pairs),
    "overlap_validation_max_error": float(max_err),
    "compilation_max_deviation": float(max_dev),
    "final_test_metrics": final.round(6).to_dict(orient="index"),
    "versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
    },
    "artifacts": {p.name: sha256(p) for p in artifacts},
}

manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

print("\nARTEFACTOS GENERADOS")
print("-" * 60)
for p in artifacts:
    print(f"  {p.name:44s} {p.stat().st_size/1024:8.1f} KB")

print(f"\nManifiesto con hashes SHA-256: {manifest_path}")
print("\nCuaderno terminado.")

RESUMEN DEL EXPERIMENTO
Dataset            : water_potability_80.csv  (80 filas usables)
Split              : 64 train / 16 test  (80/20 estratificado)
Features           : ph, Hardness, Sulfate, Conductivity
Feature map        : personalizado, 4 qubits, aristas [(0, 1), (1, 3), (2, 3)]
Kernel             : fidelidad |<phi_j|phi_i>|^2
Baseline RBF       : {'C': 1, 'gamma': 'scale'}
C del QSVM         : 1.0 (fijo)
CV                 : 20 particiones pareadas
Emulador           : H2-1LE, shots [256, 1024], 136 circuitos por presupuesto

TEST RESERVADO — cinco métricas
------------------------------------------------------------
             Accuracy  Precision  Recall     F1  Balanced_Accuracy
Model                                                             
SVM_RBF        0.6250     0.6000  0.7500 0.6667             0.6250
QSVM_Custom    0.5625     0.5714  0.5000 0.5333             0.5625

CONCLUSIÓN HONESTA
------------------------------------------------------------
Delta BA en test 

## Notas de operación (Nexus web)

**Orden de ejecución.** Si el kernel se reinicia perdés todas las variables y vas
a ver `NameError`. Usá *Run → Run All Above* o *Restart Kernel and Run All Cells*.

**El CSV.** Subilo con el botón de upload del Lab; queda en el directorio del
notebook. Si el nombre no es `water_potability_80.csv`, el Bloque 2 te lista los
CSV que encuentra para que corrijas `DATA_PATH`.

**No usar pip desde el notebook.** El entorno de Nexus viene gestionado. Si de
verdad falta algo, abrí una terminal del Lab.

### Estrategia de cuota

| Bloques | Costo | Qué producen |
|---|---|---|
| 0–9 | Cero, todo local | Baseline RBF, kernel exacto, CV pareada, heatmap |
| 10–11 | Cero (compilar no gasta shots) | Recursos 2Q compilados para H2 |
| 11B | Cero, local | Equivalencia numérica |
| **12** | **Único que gasta** | Kernel medido en el emulador |
| 13–16 | Cero, local | Ruido de muestreo, test final, auditoría |

Corré 0–11 y 11B con `H2_DEVICE = "H2-1LE"`. Cuando todo pase sin error, decidí
si el Bloque 12 va con `H2-1LE` (gratis, sin ruido físico) o con `H2-Emulator`
(consume cuota, con ruido de calibración real).

**Recorte del Bloque 12** vía `AUDIT_N` en el Bloque 1:

| `AUDIT_N` | Circuitos | Shots con [256, 1024] |
|---|---|---|
| 16 | 136 | 174 080 |
| 12 | 78 | 99 840 |
| 8 | 36 | 46 080 |

Si el presupuesto aprieta, `AUDIT_N = 12` con ambos niveles conserva la
verificación del escalamiento $S^{-1/2}$, que es el resultado defendible del
análisis de ruido. Perder ese escalamiento a cambio de cuatro muestras más no
vale la pena.

**Reenganche.** El Bloque 12 guarda el `job_id` en
`outputs_qsvm_h2/checkpoints/`. Si la sesión se cae, volvé a correr la celda: se
reengancha al job que ya está en el servidor en vez de reenviarlo. Si cambiás
`AUDIT_N` o el dispositivo, borrá esa carpeta primero — los checkpoints viejos
ya no corresponden.

### Sobre `H2-Emulator` y el informe

`H2-1LE` es un emulador **sin ruido**: la única perturbación es el muestreo
binomial finito, que es exactamente lo que modela el Bloque 13.

`H2-Emulator` trae `noise_specs` reales (fallos 1Q ≈ 2.9e-5, 2Q ≈ 1.3e-3,
dephasing, crosstalk, errores de medición). Si corrés el Bloque 12 contra ese
dispositivo, el error observado ya **no** es solo ruido de shots, y la razón
$S^{-1/2}$ puede desviarse de 2.0 justamente por eso. Es un resultado más rico y
más honesto, pero hay que nombrarlo así en la sección de limitaciones: el
Bloque 13 asume muestreo binomial ideal, y con `H2-Emulator` esa hipótesis deja
de sostenerse por sí sola.

**Con n = 80** las métricas de test tienen incertidumbre grande. El cuaderno lo
asume: reporta IQR y bootstrap en la CV, y no afirma ventaja cuántica en ningún
punto.